In [1]:
import pandas as pd
import numpy as np
 
DATA_PATH = "../data/cardekho_dataset.csv"        # notebook lives in notebooks/
OUT_PATH  = "../data/cardekho_cleaned.csv"
 
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
print("Raw shape:", df.shape)
 
def log(step, before):
    print(f"{step:<55} rows removed: {before - len(df):>4}  -> {len(df)} left")

Raw shape: (15411, 14)


## 1.1 Drop the leftover index column
# `Unnamed: 0` runs from 0 to 19543 but there are only 15,411 rows, so this is an index
# from a bigger original file. It has no meaning for prediction, and it also hides duplicates.

In [2]:
df = df.drop(columns="Unnamed: 0")

## 1.2 Remove exact duplicates

In [45]:
n = len(df)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
log("Drop exact duplicates", n)

Duplicate rows: 167
Drop exact duplicates                                   rows removed:  167  -> 15244 left


## 1.3 Fix inconsistent labels
# - `ISUZU` vs `Isuzu` (same brand, two spellings)
# - `redi-GO` vs `RediGO` (Datsun)
# - `Dzire LXI/VXI/ZXI` are variants of `Swift Dzire` (Maruti)
# - `Grand` is a truncated `Grand i10` (Hyundai)

In [3]:
df["brand"] = df["brand"].replace({"ISUZU": "Isuzu"})
df["model"] = df["model"].replace({
    "redi-GO": "RediGO",
    "Dzire LXI": "Swift Dzire",
    "Dzire VXI": "Swift Dzire",
    "Dzire ZXI": "Swift Dzire",
    "Grand": "Grand i10",
})
print("Brands:", df["brand"].nunique(), "| Models:", df["model"].nunique())

Brands: 31 | Models: 116



## 1.4 Fix impossible values
# **seats = 0** (2 rows): fill with the most common seat count of that same model.
#
# **Toyota Camry labelled "Electric"** (4 rows): Camry is sold in India as a hybrid, not an EV,
# so these are relabelled `Hybrid`. Otherwise the model would think "Electric" means a Camry.
 

In [4]:
df.loc[df["seats"] == 0, "seats"] = (
    df.groupby("model")["seats"].transform(lambda s: s[s > 0].mode()[0])
)
print("seats == 0 left:", (df["seats"] == 0).sum())
 
df.loc[(df["model"] == "Camry") & (df["fuel_type"] == "Electric"), "fuel_type"] = "Hybrid"
print(df["fuel_type"].value_counts())

seats == 0 left: 0
fuel_type
Petrol    7643
Diesel    7419
CNG        301
LPG         44
Hybrid       4
Name: count, dtype: int64



# ## 1.5 km_driven errors
# - `km_driven > 500,000` is physically unrealistic for these cars (some show 3.8 million km in 5 years).
# - In 22 rows `km_driven` is *exactly equal* to `selling_price` (e.g. Innova: 950,000 km and 950,000 rupees).
#   That is a data-entry mistake (price typed into the km column). Small coincidences like an old Alto at
#   120,000 km and 120,000 rupees are plausible, so only the large ones (>= 300,000) are removed.

In [6]:
n = len(df)
df = df[df["km_driven"] <= 500_000]
log("Drop km_driven > 500,000", n)
 
n = len(df)
same = (df["km_driven"] == df["selling_price"]) & (df["km_driven"] >= 300_000)
df = df[~same].reset_index(drop=True)
log("Drop km_driven == selling_price (large values)", n)

Drop km_driven > 500,000                                rows removed:   12  -> 15399 left
Drop km_driven == selling_price (large values)          rows removed:    2  -> 15397 left


## 1.6 Age and price anomalies
# - `vehicle_age >= 25`: a 29-year-old Maruti Alto (the Alto launched in 2000) and a 25-year-old BMW 3
#   priced at 10 lakh are not believable.
# - A 1-year-old Toyota Camry priced at Rs 3.45 lakh (new price is 30+ lakh) is a price typo.
 

In [7]:
n = len(df)
df = df[df["vehicle_age"] < 25]
log("Drop vehicle_age >= 25", n)
 
n = len(df)
bad_camry = (df["model"] == "Camry") & (df["vehicle_age"] <= 1) & (df["selling_price"] < 1_000_000)
df = df[~bad_camry].reset_index(drop=True)
log("Drop 1-year-old Camry at Rs 3.45 lakh", n)

Drop vehicle_age >= 25                                  rows removed:    2  -> 15395 left
Drop 1-year-old Camry at Rs 3.45 lakh                   rows removed:    1  -> 15394 left


## 1.7 Drop redundant column
# `car_name` is always `brand + " " + model`, so it adds nothing.

In [8]:
df = df.drop(columns="car_name")

## 1.8 Final checks and save


In [9]:
print("Final shape:", df.shape, "| removed", len(df_raw) - len(df), "of", len(df_raw), "rows")
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
df.describe().T

Final shape: (15394, 12) | removed 17 of 15411 rows
Missing values: 0
Duplicates: 167


,count,mean,std,min,25%,50%,75%,max
vehicle_age,15394.0,6.034039,3.004427,0.0,4.0,6.00,8.0,22.00
km_driven,15394.0,54804.012667,36358.422153,100.0,30000.0,50000.00,70000.0,500000.00
mileage,15394.0,19.702993,4.170759,4.0,17.0,19.67,22.7,33.54
engine,15394.0,1485.779719,520.946101,793.0,1197.0,1248.00,1582.0,6592.00
max_power,15394.0,100.573457,42.972774,38.4,74.0,88.50,117.3,626.00
seats,15394.0,5.326101,0.805400,2.0,5.0,5.00,5.0,9.00
selling_price,15394.0,774954.324737,894387.057157,40000.0,385000.0,556000.00,824250.0,39500000.00


In [10]:
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Saved: ../data/cardekho_cleaned.csv


In [11]:
df.head()

,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai,Grand i10,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
